# OTTO Multi-Objective Recommender System


<img src='https://media.lesechos.com/api/v1/images/view/5c0649e93e454678216c0610/1280x720/021990291944-web.jpg'>


Bu çalışmada OTTO Multi-Objective Recommender System yarışması için session bazlı, recent davranış ve ürün sıklığını birlikte kullanan bir recommendation baseline oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip açma
3. Veri dosyalarını tanımlama
4. Train ve test örneklerini yükleme
5. Session yapısını anlama
6. Global popüler ürünleri çıkarma
7. Session bazlı recommendation mantığı
8. Submission oluşturma
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [ ]:
# Bu bölümde jsonl veri yapısını okuyup session bazlı recommendation baseline kurmak için gerekli kütüphaneleri içe aktarıyoruz.


In [ ]:
from google.colab import drive
import os
import zipfile
import json
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


In [2]:
# Bu bölümde Google Drive içindeki yarışma zip dosyasını Colab çalışma alanına bağlayıp dosyaları doğrudan zip içinden okuyacağız.


In [ ]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/otto-recommender-system.zip'


In [ ]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/otto-recommender-system.zip'
extract_dir = '/content/otto_recommender_system'

os.makedirs(extract_dir, exist_ok=True)

expected_files = {'train.jsonl', 'test.jsonl', 'sample_submission.csv'}
existing_files = set()
for root, _, files in os.walk(extract_dir):
    existing_files.update(files)

if not expected_files.issubset(existing_files):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extract dir: /content/otto_recommender_system
Top-level: ['train.jsonl', 'sample_submission.csv', 'test.jsonl']


In [ ]:
def find_file(root_dir, filename):
    for root, _, files in os.walk(root_dir):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f'{filename} bulunamadi.')

train_path = find_file(extract_dir, 'train.jsonl')
test_path = find_file(extract_dir, 'test.jsonl')
sample_submission_path = find_file(extract_dir, 'sample_submission.csv')


In [ ]:
sample_train_sessions = 120000
sample_test_sessions = 30000

def load_jsonl_sample(file_path, limit):
    rows = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, raw_line in enumerate(f):
            if i >= limit:
                break
            rows.append(json.loads(raw_line))
    return pd.DataFrame(rows)

train = load_jsonl_sample(train_path, sample_train_sessions)
test = load_jsonl_sample(test_path, sample_test_sessions)
sample_submission = pd.read_csv(sample_submission_path)

print('Train shape (sample):', train.shape)
print('Test shape (sample):', test.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


In [ ]:
sample_train_sessions = 120000
sample_test_sessions = 30000

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()


def find_zip_member(members, filename):
    for member in members:
        if member.endswith(filename):
            return member
    raise FileNotFoundError(f'{filename} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, 'train.jsonl')
test_member = find_zip_member(zip_members, 'test.jsonl')
sample_submission_member = find_zip_member(zip_members, 'sample_submission.csv')


def load_jsonl_sample_from_zip(zip_path, member_name, limit):
    rows = []
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            for i, raw_line in enumerate(f):
                if i >= limit:
                    break
                rows.append(json.loads(raw_line.decode('utf-8')))
    return pd.DataFrame(rows)


def load_csv_from_zip(zip_path, member_name):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            return pd.read_csv(f)

train = load_jsonl_sample_from_zip(zip_path, train_member, sample_train_sessions)
test = load_jsonl_sample_from_zip(zip_path, test_member, sample_test_sessions)
sample_submission = load_csv_from_zip(zip_path, sample_submission_member)

print('Train shape (sample):', train.shape)
print('Test shape (sample):', test.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape (sample): (120000, 2)
Test shape (sample): (30000, 2)
Sample submission shape: (5015409, 2)


session  \
0        0   
1        1   
2        2   
3        3   
4        4   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

## 5. Session Yapısını Anlama


In [ ]:
# Bu bölümde her session içinde event listesinin nasıl tutulduğunu inceliyoruz.


In [ ]:
print('Ilk session örneği:')
print(train.iloc[0]['session'])
print(train.iloc[0]['events'][:3])


Ilk session örneği:
0
[{'aid': 1517085, 'ts': 1659304800025, 'type': 'clicks'}, {'aid': 1563459, 'ts': 1659304904511, 'type': 'clicks'}, {'aid': 1309446, 'ts': 1659367439426, 'type': 'clicks'}]


In [ ]:
def flatten_events(df):
    rows = []
    for _, row in df.iterrows():
        session = row['session']
        for event in row['events']:
            rows.append({
                'session': session,
                'aid': event.get('aid'),
                'ts': event.get('ts'),
                'type': event.get('type')
            })
    return pd.DataFrame(rows)

train_events = flatten_events(train)
test_events = flatten_events(test)

print('Train events shape:', train_events.shape)
print('Test events shape:', test_events.shape)
train_events.head()


Train events shape: (6278759, 4)
Test events shape: (156183, 4)


,session,aid,ts,type
0,0,1517085,1659304800025,clicks
1,0,1563459,1659304904511,clicks
2,0,1309446,1659367439426,clicks
3,0,16246,1659367719997,clicks
4,0,1781822,1659367871344,clicks


In [ ]:
type_map = {
    0: 'clicks',
    1: 'carts',
    2: 'orders',
    'clicks': 'clicks',
    'carts': 'carts',
    'orders': 'orders'
}

train_events['event_type'] = train_events['type'].map(type_map).fillna(train_events['type'])
test_events['event_type'] = test_events['type'].map(type_map).fillna(test_events['type'])

train_events[['session', 'aid', 'ts', 'event_type']].head()


,session,aid,ts,event_type
0,0,1517085,1659304800025,clicks
1,0,1563459,1659304904511,clicks
2,0,1309446,1659367439426,clicks
3,0,16246,1659367719997,clicks
4,0,1781822,1659367871344,clicks


## 6. Global Popüler Ürünleri Çıkarma


In [ ]:
# Bu bölümde tüm örnek train verisinde en sık görülen ürünleri çıkarıyoruz. Bunlar öneri listemizi tamamlamak için kullanılacak.


In [ ]:
top_clicks = train_events.loc[train_events['event_type'] == 'clicks', 'aid'].value_counts().head(20).index.tolist()
top_carts = train_events.loc[train_events['event_type'] == 'carts', 'aid'].value_counts().head(20).index.tolist()
top_orders = train_events.loc[train_events['event_type'] == 'orders', 'aid'].value_counts().head(20).index.tolist()
top_global = train_events['aid'].value_counts().head(20).index.tolist()

print('Top clicks:', top_clicks[:10])
print('Top carts:', top_carts[:10])
print('Top orders:', top_orders[:10])


Top clicks: [29735, 832192, 1603001, 108125, 1733943, 554660, 80222, 1460571, 166037, 1083665]
Top carts: [166037, 29735, 485256, 80222, 1733943, 1672890, 832192, 1022566, 1498443, 152547]
Top orders: [80222, 1022566, 166037, 351335, 1629608, 1733943, 1603001, 332654, 231487, 1083665]


## 7. Session Bazlı Recommendation Mantığı


In [ ]:
# Bu bölümde her session için recent behavior (yakın davranış) ve frequency (sıklık) bilgisini birlikte kullanıyoruz.


In [ ]:
def unique_preserve_order(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


def build_candidates_for_session(session_df, target):
    session_df = session_df.sort_values('ts')
    recent_aids = session_df['aid'].tolist()[::-1]
    recent_unique = unique_preserve_order(recent_aids)

    freq_all = session_df['aid'].value_counts().index.tolist()
    freq_target = session_df.loc[session_df['event_type'] == target, 'aid'].value_counts().index.tolist()

    if target == 'clicks':
        base_candidates = recent_unique + freq_all + top_clicks + top_global
    elif target == 'carts':
        base_candidates = freq_target + recent_unique + top_carts + top_global
    else:
        order_like = session_df.loc[session_df['event_type'].isin(['orders', 'carts']), 'aid'].value_counts().index.tolist()
        base_candidates = order_like + recent_unique + top_orders + top_global

    final_candidates = []
    seen = set()
    for aid in base_candidates:
        if aid not in seen:
            seen.add(aid)
            final_candidates.append(str(aid))
        if len(final_candidates) == 20:
            break
    return ' '.join(final_candidates)


In [ ]:
session_predictions = []

for session_id, session_df in test_events.groupby('session'):
    click_preds = build_candidates_for_session(session_df, 'clicks')
    cart_preds = build_candidates_for_session(session_df, 'carts')
    order_preds = build_candidates_for_session(session_df, 'orders')

    session_predictions.append((f'{session_id}_clicks', click_preds))
    session_predictions.append((f'{session_id}_carts', cart_preds))
    session_predictions.append((f'{session_id}_orders', order_preds))

predictions_df = pd.DataFrame(session_predictions, columns=['session_type', 'labels'])
predictions_df.head()


,session_type,labels
0,12899779_clicks,59625 29735 832192 1603001 108125 1733943 554660 80222 1460571 166037 1083665 184976 959208 247240 332654 1743151 1116095 673407 714524 508883
1,12899779_carts,59625 166037 29735 485256 80222 1733943 1672890 832192 1022566 1498443 152547 1603001 544144 554660 1083665 1629608 1116095 332654 351335 1743151
2,12899779_orders,59625 80222 1022566 166037 351335 1629608 1733943 1603001 332654 231487 1083665 923948 544144 832192 29735 673407 1445562 800391 563117 1406660
3,12899780_clicks,1142000 736515 973453 582732 29735 832192 1603001 108125 1733943 554660 80222 1460571 166037 1083665 184976 959208 247240 332654 1743151 1116095
4,12899780_carts,1142000 736515 973453 582732 166037 29735 485256 80222 1733943 1672890 832192 1022566 1498443 152547 1603001 544144 554660 1083665 1629608 1116095


## 8. Submission Oluşturma


In [ ]:
# Bu bölümde örnek submission formatına uygun tahmin dosyasını oluşturuyoruz.


In [ ]:
submission = sample_submission[['session_type']].copy()
submission = submission.merge(predictions_df, on='session_type', how='left')

fallback_labels = ' '.join(map(str, top_global[:20]))
submission['labels'] = submission['labels'].fillna(fallback_labels)

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (5015409, 2)


,session_type,labels
0,12899779_clicks,59625 29735 832192 1603001 108125 1733943 554660 80222 1460571 166037 1083665 184976 959208 247240 332654 1743151 1116095 673407 714524 508883
1,12899779_carts,59625 166037 29735 485256 80222 1733943 1672890 832192 1022566 1498443 152547 1603001 544144 554660 1083665 1629608 1116095 332654 351335 1743151
2,12899779_orders,59625 80222 1022566 166037 351335 1629608 1733943 1603001 332654 231487 1083665 923948 544144 832192 29735 673407 1445562 800391 563117 1406660
3,12899780_clicks,1142000 736515 973453 582732 29735 832192 1603001 108125 1733943 554660 80222 1460571 166037 1083665 184976 959208 247240 332654 1743151 1116095
4,12899780_carts,1142000 736515 973453 582732 166037 29735 485256 80222 1733943 1672890 832192 1022566 1498443 152547 1603001 544144 554660 1083665 1629608 1116095


In [ ]:
output_path = '/content/otto_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/otto_submission.csv


## 9. Sonuç


Bu çalışmada OTTO Multi-Objective Recommender System yarışması için kullanıcı oturumları incelenerek ürün önerisi yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçta son etkileşimler, sık görülen ürünler ve genel popüler ürünler birlikte değerlendirilerek tıklama, sepete ekleme ve sipariş için öneri listeleri üretildi.
